# MedLook-4B — Colab Train + Eval Notebook

> **Research prototype only. Not for clinical use, diagnosis, or treatment decisions.**

This notebook is the **only** place real training/evaluation happens. Your local
machine (see `README.md`) is for pipeline validation only (fixtures, `--dry-run`,
`--mock`, `--no-weights`) -- never for training the real 4B model.

## Prerequisites (do these before running anything below)

1. A Colab **GPU runtime**: L4 or A100 preferred (Runtime -> Change runtime type). This
   will not fit comfortably on a T4.
2. A Hugging Face account with the **MedGemma license accepted**
   (https://huggingface.co/google/medgemma-1.5-4b-it) and an access token.
3. This repository pushed somewhere Colab can fetch it (GitHub, or uploaded to Drive).
4. (Optional, needed for strategy/calibration eval) A hand-labeled
   `data/gold_strategy_set.json` + `data/gold_strategy_set_images/` -- see
   `medlook/data/gold_strategy_set.py` for the labeling instructions. Without this,
   training and answer-quality eval still work; strategy/calibration eval is skipped.

## What this notebook does, in order

1. Mount Drive, install dependencies, log in to Hugging Face.
2. Prepare data (writes to Drive so it survives session restarts).
3. Run a multi-image packing smoke test on ~50 real samples (catches the #1 way to
   waste Colab compute, in under a minute).
4. Train Short-SFT and Full-MedLook (Process-SFT is optional).
5. Generate real predictions and run the four-system evaluation report.
6. (Optional) Export to 16-bit / GGUF, and launch the Gradio demo in-notebook.

Read the printed `success_gate` verdict honestly -- do not re-run with different seeds
looking for a pass, and do not soften a "NOT PASSED" result when reporting it.

## 1. Check GPU, mount Drive, clone the repo, install dependencies

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# EDIT THIS: point at wherever you pushed this repo (GitHub URL, or a Drive path you
# already uploaded to). If you already have it in Drive, skip the git clone and just
# set REPO_DIR to that path.
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_MEDLOOK_REPO.git"
REPO_DIR = "/content/medlook2"

import os

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

In [ ]:
%pip install -q -r requirements.txt -r requirements-colab.txt
%pip install -q -e . --no-deps

In [ ]:
import getpass

from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN") or getpass.getpass(
    "Hugging Face token (must have the MedGemma license accepted): "
)
login(hf_token)

## 2. Prepare data

The repo's `configs/*.yaml` default to local `output_dir`s so local pipeline
validation never touches Drive. On Colab we rewrite `data.output_dir` and
`train.output_dir` to Drive-mounted paths so prepared datasets and checkpoints survive
a session disconnect -- the single most common way to lose hours of work on Colab.

**Minimal-viable-experiment first:** before committing to a full run, set `hf_limit`
in each config's `open_vqa`/`meissa` sections to ~700 (giving a 2-4k sample mix) and
confirm Short-SFT vs Full-MedLook behave sanely before scaling up to `hf_limit: null`.

In [ ]:
import yaml

DRIVE_ROOT = "/content/drive/MyDrive/medlook_runs"
os.makedirs(DRIVE_ROOT, exist_ok=True)


def make_colab_config(src_path: str, dst_path: str) -> str:
    with open(src_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    name = cfg["name"]
    cfg["data"]["output_dir"] = f"{DRIVE_ROOT}/data/{name}"
    cfg["train"]["output_dir"] = f"{DRIVE_ROOT}/checkpoints/{name}"
    with open(dst_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return dst_path


SHORT_SFT_CFG = make_colab_config("configs/short_sft.yaml", "configs/short_sft.colab.yaml")
PROCESS_SFT_CFG = make_colab_config("configs/process_sft.yaml", "configs/process_sft.colab.yaml")
FULL_MEDLOOK_CFG = make_colab_config("configs/full_medlook.yaml", "configs/full_medlook.colab.yaml")
print(SHORT_SFT_CFG, PROCESS_SFT_CFG, FULL_MEDLOOK_CFG)

In [ ]:
# All three configs share the same underlying data mix and only differ in `profile`
# (see the header comment in each configs/*.yaml) -- running prepare_data three times
# with the same curriculum seed keeps the same underlying examples across ablations,
# so the four-system comparison isolates the effect of the output schema itself.
!python scripts/prepare_data.py --config {SHORT_SFT_CFG}
!python scripts/prepare_data.py --config {PROCESS_SFT_CFG}
!python scripts/prepare_data.py --config {FULL_MEDLOOK_CFG}

## 3. Multi-image packing smoke test (do this before any full training run)

Runs the real base model + the real `UnslothVisionDataCollator` on the first 50
prepared training samples, without starting a training loop. Multi-image packing
failures are the single most likely way to waste hours of Colab compute -- this
catches them in under a minute.

In [ ]:
!python scripts/train.py --config {FULL_MEDLOOK_CFG} --packing-smoke-test 50

## 4. Train the ablations

Each run saves checkpoints every `save_steps` to Drive (`train.output_dir` was
rewritten above) and supports `--resume-from-checkpoint <path>` if a session drops.

Train **Short-SFT** and **Full-MedLook** at minimum -- these two are enough to check
the primary success gate. **Process-SFT** is optional and only needed for the complete
four-system table; run it if time/compute allows.

In [ ]:
# Short-SFT (baseline SFT ablation: [FINAL] only, no strategy/process).
!python scripts/train.py --config {SHORT_SFT_CFG}

In [ ]:
# Full-MedLook (primary system: [STRATEGY] + [PROCESS] + [FINAL]).
!python scripts/train.py --config {FULL_MEDLOOK_CFG}

In [ ]:
# Process-SFT (optional: [PROCESS] + [FINAL], no [STRATEGY] -- isolates the process
# block's contribution separately from the strategy block's).
!python scripts/train.py --config {PROCESS_SFT_CFG}

## 5. Generate real predictions, then run the four-system evaluation report

`--split` below MUST be a split that was **not** used for training (`prepare_data.py`
defaults to `hf_split: train`). Verify the real split names available for
vqa_rad/pathvqa/slake on the Hugging Face Hub before relying on `"test"` -- this
notebook does not invent a held-out split if the requested one doesn't exist upstream,
the command will simply fail loudly.

If you haven't authored `data/gold_strategy_set.json` yet, the cell below skips
strategy/calibration eval automatically and only produces the answer-quality
comparison.

In [ ]:
PRED_DIR = f"{DRIVE_ROOT}/predictions"
EVAL_SPLIT = "test"  # VERIFY this split exists for vqa_rad/pathvqa/slake before relying on it

GOLD_ARGS = ""
if os.path.exists("data/gold_strategy_set.json"):
    GOLD_ARGS = (
        "--gold-strategy-json data/gold_strategy_set.json "
        "--gold-strategy-image-dir data/gold_strategy_set_images"
    )
else:
    print(
        "WARNING: data/gold_strategy_set.json not found -- skipping strategy/calibration "
        "eval. See medlook/data/gold_strategy_set.py for labeling instructions."
    )

CHECKPOINTS = f"{DRIVE_ROOT}/checkpoints"

In [ ]:
# Base: no adapter -- uses the plain pretrained model.
!python scripts/generate_predictions.py --config {FULL_MEDLOOK_CFG} \
    --system-name base --split {EVAL_SPLIT} --out-dir {PRED_DIR} {GOLD_ARGS}

In [ ]:
!python scripts/generate_predictions.py --config {SHORT_SFT_CFG} \
    --adapter-dir {CHECKPOINTS}/short_sft/final_adapter \
    --system-name short_sft --split {EVAL_SPLIT} --out-dir {PRED_DIR} {GOLD_ARGS}

In [ ]:
# Optional: only if you also trained Process-SFT above.
!python scripts/generate_predictions.py --config {PROCESS_SFT_CFG} \
    --adapter-dir {CHECKPOINTS}/process_sft/final_adapter \
    --system-name process_sft --split {EVAL_SPLIT} --out-dir {PRED_DIR} {GOLD_ARGS}

In [ ]:
!python scripts/generate_predictions.py --config {FULL_MEDLOOK_CFG} \
    --adapter-dir {CHECKPOINTS}/full_medlook/final_adapter \
    --system-name full_medlook --split {EVAL_SPLIT} --out-dir {PRED_DIR} {GOLD_ARGS}

In [ ]:
# The real four-system report -- NOT --mock. Read the printed success_gate verdict
# honestly (see PROJECT_BLUEPRINT.md / BUILD_PLAN.md for what it means and what to do
# if it says NOT PASSED: diagnose via the confusion matrix in the JSON report, don't
# just re-run hoping for a different seed).
!python scripts/eval.py --predictions-dir {PRED_DIR} --out {DRIVE_ROOT}/report.json

## 6. (Optional) Export the winning system

Merges the LoRA adapter into a standalone 16-bit model, then attempts a GGUF export
for CPU inference. GGUF export for vision-language architectures is less mature than
for text-only models -- if it fails, fall back to running llama.cpp's
`convert_hf_to_gguf.py` directly against the merged 16-bit directory (see
`medlook/export/gguf.py`'s docstring for the full caveat).

In [ ]:
from medlook.export.gguf import export_gguf
from medlook.export.merge import merge_lora_to_fp16

WINNING_CFG = FULL_MEDLOOK_CFG  # change if a different ablation actually won the eval
WINNING_ADAPTER = f"{CHECKPOINTS}/full_medlook/final_adapter"
EXPORT_DIR = f"{DRIVE_ROOT}/exports"

merged_dir = merge_lora_to_fp16(WINNING_CFG, WINNING_ADAPTER, f"{EXPORT_DIR}/merged_16bit")
print("Merged 16-bit model at:", merged_dir)

try:
    gguf_dir = export_gguf(WINNING_CFG, WINNING_ADAPTER, f"{EXPORT_DIR}/gguf", quantization="q4_k_m")
    print("GGUF export at:", gguf_dir)
except Exception as exc:
    print("GGUF export failed or is unsupported for this architecture right now:", exc)
    print("Fall back: run llama.cpp's convert_hf_to_gguf.py directly against", merged_dir)

## 7. (Optional) Launch the Gradio demo in-notebook, with real weights

In [ ]:
from medlook.demo.gradio_app import build_demo
from medlook.train.sft import load_model_with_optional_adapter

model, tokenizer = load_model_with_optional_adapter(WINNING_CFG, WINNING_ADAPTER)
demo = build_demo(model=model, tokenizer=tokenizer)
demo.launch(share=True)

## Wrap-up: honesty checklist before reporting results

- [ ] Did `scripts/eval.py` run WITHOUT `--mock`? (mock output must never be quoted)
- [ ] Does the report's `success_gate.passed` field say what you are about to claim?
- [ ] If `success_gate.passed` is `false`, are you reporting that honestly rather than
      cherry-picking a favorable metric?
- [ ] Have you filled in `model_card_template.md` with these exact numbers (copy from
      the JSON report, don't retype/round by hand)?
- [ ] Have you included a few real, unedited qualitative examples (not just aggregate
      metrics) covering at least one RELOOK and one FLAG_UNCERTAIN/ESCALATE case?

If the success gate did not pass: do not invent an improvement. Diagnose using the
per-class precision/recall and confusion matrix in the JSON report (strategy
confusion, class imbalance, multi-image packing, learning rate, data mix are the usual
suspects -- see PROJECT_BLUEPRINT.md's risk analysis section) and propose a concrete
next experiment.